## Purpose
Make 

In [2]:
import pandas as pd
import numpy as np

In [5]:
# just get the two excel file (synthetic terminator + nature terminator) and merge them together, then save as a new excel file
df_natural = pd.read_excel("Chen_2013_terminators/41592_2013_BFnmeth2515_MOESM206_ESM.xlsx")
df_synthetic = pd.read_excel("Chen_2013_terminators/41592_2013_BFnmeth2515_MOESM207_ESM.xlsx")


c:\venvs\TE_model\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [6]:
print(df_natural.shape)
print(df_synthetic.shape)

(317, 29)
(265, 28)


In [7]:
df_thetwo = df_natural[df_natural["Name"].isin(["pheA-1", "ECK120034435"])]
phe = df_thetwo[df_thetwo["Name"] == "pheA-1"]
eck = df_thetwo[df_thetwo["Name"] == "ECK120034435"]
pd.set_option("display.max_colwidth", None) #turn off the maximum width 50 nt set by the pandas display option by default
print(phe["Sequence"])
print(phe["A-tract"])
print(phe["Hairpin"])
print(phe["U-tract"])
print(phe["Structure"])
print("eck===========")
print(eck["Sequence"])
print(eck["A-tract"])
print(eck["Hairpin"])
print(eck["U-tract"])
print(eck["Structure"])

# I will do the swap first now, and also when I do the stuff, I need to check whether all the A-tract, Hairpin, U-tract are part of the substring of the sequence I get (up flank + sequence + down flank).

2    gacgaacaaTAAGGCCTCCCAAATCGGGGGGCCTTTTTTATTgaTaacaaaa
Name: Sequence, dtype: str
2    AGAAAAGA
Name: A-tract, dtype: str
2    GACGCUGAAAAGCGUC
Name: Hairpin, dtype: str
2    UUUUUUCGUUUU
Name: U-tract, dtype: str
2    ((((((....))))))
Name: Structure, dtype: str
eck===========
3    CTCGGTACCAAATTCCAGAAAAGAGACGCTGAAAAGCGTCTTTTTTCGTTTTGGTCC
Name: Sequence, dtype: str
3    AACAAUAA
Name: A-tract, dtype: str
3    GGCCUCCCAAAUCGGGGGGCC
Name: Hairpin, dtype: str
3    UUUUUUAUUGAU
Name: U-tract, dtype: str
3    ((((((((.....))))))))
Name: Structure, dtype: str


In [ ]:
#save flanking region sequences everything is 5' to 3' on the plasmid. 
upstream_flank = "ccttcttgagtttgtaacagctgctgggattacacatggcatggatgaactatacaaataataagaattc" #note that only the end 6 nt belongs to "nothing sequence", those ahead of it are part of GFP
print(len(upstream_flank)) #70 which is also the index to access the insert
downstream_flank = "actagtagcggccgctgcagaaagaggagaaatactagatggcttcctccgaagacgttatcaaagagttcatg" #here, 38nt belongs to nothing, then the rest downstream are part of RFP 
print(len(downstream_flank)) #74
#change the T to U:
upstream_flank_u = upstream_flank.replace("t", "u").upper()
downstream_flank_u = downstream_flank.replace("t", "u").upper()


70
74


In [25]:
# --- concatenate the two sheets ---
# The sheets spell delta differently: natural uses U+0394 and synthetic U+2206.
# Without renaming, pd.concat produces BOTH sets of columns, each half-empty
# (36 columns instead of 29). Normalise the names first.
def normalise_delta(d):
    return d.rename(columns={c: "d" + str(c)[1:] for c in d.columns
                             if str(c)[:1] in ("Δ", "∆")}) #turned all the delta to d; confirmed that only column titles have delta

df_full = pd.concat([normalise_delta(df_natural).assign(library="natural"),
                     normalise_delta(df_synthetic).assign(library="synthetic")],
                    ignore_index=True)
print("columns:", df_full.shape[1], "(29 data + library; not 36)")
#confirmed comparing to previous reproducing xgboost, the size is right and column headers right


columns: 31 (29 data + library; not 36)


In [26]:
for i, c in enumerate(df_full.columns):
    print(f"[{i:2d}] {c}")

[ 0] Name
[ 1] Recombination-resistant strong terminators
[ 2] Recombination-resistant medium terminators
[ 3] Scaffold 1
[ 4] Transcription Unit
[ 5] Sequence
[ 6] Average Uninduced GFP
[ 7] Average Induced GFP
[ 8] Average Uninduced RFP
[ 9] Average Induced RFP
[10] Average Induced RFP.1
[11] Average Strength
[12] Standard Deviation of Uninduced GFP
[13] Standard Deviation of Induced GFP
[14] Standard Deviation of Uninduced RFP
[15] Standard Deviation of Induced RFP
[16] Standard Deviation of GFP/RFP
[17] Standard Deviation of Strength
[18] A-tract
[19] Hairpin
[20] Structure
[21] Loop
[22] U-tract
[23] dGU
[24] dGL
[25] dGH
[26] dGA
[27] dGB
[28] Predicted TS
[29] library
[30] Average GFP/RFP


In [27]:
df_full["library"] #okay claude added this one as indicating what type of terminator it is. Shit already hard to understand claude's code......

0        natural
1        natural
2        natural
3        natural
4        natural
         ...    
577    synthetic
578    synthetic
579    synthetic
580    synthetic
581    synthetic
Name: library, Length: 582, dtype: str

In [28]:
# --- swap the annotation blocks of pheA-1 and ECK120034435 ---
# These two adjacent rows carry each other's annotations (off-by-one in the supplement).
# Evidence: each one's listed Hairpin occurs in the OTHER's Sequence, and the "Scaffold 1"
# column marks pheA-1 together with the L3S1* family, whose hairpin is the one currently
# filed under ECK120034435.
# Swap ONLY the annotation columns. Name / Sequence / fluorescence stay put, because the
# strength was measured on the construct, and Name+Sequence identify the construct.

SWAP_COLS = ["A-tract", "Hairpin", "Structure", "Loop", "U-tract",
             "dGU", "dGL", "dGH", "dGA", "dGB", "Predicted TS"] 
# TODO I am unsure of the free energies that whether they are swapped. I don't have time to look into kinefold. 
# Since we don't use these stuff for now, I will just leave as it is, so swap together with the intrinsic 
# parts prediction

i_phe = df_full.index[df_full["Name"] == "pheA-1"][0]
i_eck = df_full.index[df_full["Name"] == "ECK120034435"][0]

# Verified on pandas 3.0.2: for a SINGLE-row .loc assignment the right-hand side is a
# Series indexed by COLUMN name, and those already match the target columns -- so no
# .values is needed and no alignment problem arises. .loc returns a copy, so tmp is a
# genuine snapshot (.copy() below is belt-and-braces).
#
# Do NOT "improve" this by adding .values: .values returns a VIEW into the underlying
# array, so tmp would alias the row overwritten on the next line and the swap breaks.
# .values IS required for MULTI-row assignment, where row labels must line up -- the
# opposite situation. Worked examples: Kaggle_Pandas_course/pandasNotesChenData.ipynb.
tmp = df_full.loc[i_phe, SWAP_COLS].copy()
df_full.loc[i_phe, SWAP_COLS] = df_full.loc[i_eck, SWAP_COLS]
df_full.loc[i_eck, SWAP_COLS] = tmp

# verify the swap did what we wanted
for name in ["pheA-1", "ECK120034435"]:
    r = df_full.loc[df_full["Name"] == name].iloc[0]
    inside = r["Hairpin"].upper().replace("U", "T") in r["Sequence"].upper()
    print(f"{name:<16} hairpin={r['Hairpin']:<24} in its own Sequence: {inside}")


pheA-1           hairpin=GGCCUCCCAAAUCGGGGGGCC    in its own Sequence: True
ECK120034435     hairpin=GACGCUGAAAAGCGUC         in its own Sequence: True


In [ ]:
# change sequence to RNA (T->U)

seq = []
for _, r in df_full.iterrows(): #the first one is index
    seq.append(r["Sequence"].strip().upper().replace("T", "U"))
df_full["RNA-Sequence"] = seq

# vectorised equivalent (note .str.upper(), not .upper() -- a Series has no .upper()):
# df_full["RNA-Sequence"] = df_full["Sequence"].str.strip().str.upper().str.replace("T", "U")

In [ ]:
# build full construct 
full = []
for _, r in df_full.iterrows():
    full.append(upstream_flank_u + r["RNA-Sequence"] + downstream_flank_u)
df_full["full"] = full

# guard: nothing but nucleotides may reach RNA-FM. A stray character does NOT change
# the window length (it is one of the 90), so a len() check will not catch it.
alphabet = set("".join(df_full["full"]))
assert alphabet <= set("ACGU"), f"non-nucleotide characters in full: {alphabet - set('ACGU')}"
print("full construct alphabet OK:", sorted(alphabet))


In [61]:
# # plan: anchor on the hairpin, then do hairpin + 12 as TTS
# TTS_idx = []
# for _, r in df_full.iterrows():

TTS_index = []
window_upstream = []

for _, r in df_full.iterrows():
    # r = df_full.loc[df_full["Name"] =="pheA-1"].iloc[0]
    # .index() raises if absent (better than .find() returning -1 silently),
    # but it returns the FIRST match -- so check the hairpin is unique first.
    assert r["full"].count(r["Hairpin"]) == 1, f"hairpin not unique in {r['Name']}"
    anchor = r["full"].index(r["Hairpin"]) #index of the start of the substring
    TTS_idx = anchor + len(r["Hairpin"]) + len(r["U-tract"]) - 1
    # print(len(r["A-tract"]) + len(r["Hairpin"]) + len(r["U-tract"]))
    # print(TTS_idx)
    # print(r["full"][ :TTS_idx + 1]) #the last index is not included, so +1
    # print(len(r["full"][ :TTS_idx]))
    #then access to the part from TTS to 90th nt
    # a negative start would silently be read as "count from the end" and give a
    # 90 nt window from the wrong region, so fail loudly instead.
    assert TTS_idx + 1 - 90 >= 0, f"not enough upstream sequence for {r['Name']}"
    window_up = r["full"][TTS_idx + 1 - 90: TTS_idx + 1]
    # print(window)
    # print(len(window))
    TTS_index.append(TTS_idx)
    window_upstream.append(window_up)
df_full["TTS_index"] = TTS_index
df_full["window_upstream"] = window_upstream

# post-checks
assert (df_full["window_upstream"].str.len() == 90).all(), "not all windows are 90 nt"
assert set("".join(df_full["window_upstream"])) <= set("ACGU"), "non-nucleotide in a window"
assert all(w.endswith(u) for w, u in zip(df_full["window_upstream"], df_full["U-tract"])),     "a window does not end on its own U-tract"
print(f"{len(df_full)} windows, all 90 nt, all ending on their own U-tract")


582 windows, all 90 nt, all ending on their own U-tract


In [62]:
# --- second option: 90 nt upstream + 19 nt downstream of the TTS (109 nt total) ---
# Same anchor, just a wider slice. Kept as a separate column so both variants can be
# embedded and compared without re-running the anchoring.
DOWN = 19
window_updown = []
for _, r in df_full.iterrows():
    t = r["TTS_index"]
    assert t + 1 + DOWN <= len(r["full"]), f"not enough downstream sequence for {r['Name']}"
    window_updown.append(r["full"][t + 1 - 90 : t + 1 + DOWN])
df_full["window_up90_dn19"] = window_updown

assert (df_full["window_up90_dn19"].str.len() == 90 + DOWN).all(), "not all windows are 109 nt"
assert set("".join(df_full["window_up90_dn19"])) <= set("ACGU"), "non-nucleotide in a window"
# the 90 nt variant must be a prefix of the 109 nt variant -- same anchor, same start
assert all(b.startswith(a) for a, b in zip(df_full["window_upstream"],
                                           df_full["window_up90_dn19"])), "windows disagree"
print(f"{len(df_full)} windows of {90 + DOWN} nt; the 90 nt variant is a prefix of each")


582 windows of 109 nt; the 90 nt variant is a prefix of each


In [65]:
# --- export the training tables ---
import numpy as np
from pathlib import Path

OUT = Path("probing/data")
OUT.mkdir(parents=True, exist_ok=True)

ts = pd.to_numeric(df_full["Average Strength"], errors="coerce")
te = 1 - 1 / ts                       # negative where T_S <= 1
logit_te = np.log(ts - 1)             # ln(T_S - 1); NaN where T_S <= 1, np.log warns

base = pd.DataFrame({
    "name":          df_full["Name"],
    "library":       df_full["library"],       # natural / synthetic -- report as a stratum
    "hairpin":       df_full["Hairpin"],       # REQUIRED to build leakage-safe split groups
    "strength":      ts,                       # Chen's T_S, the raw measurement
    "te":            te.where(ts > 1),         # NaN rather than a negative "probability"
    "logit_te":      logit_te,                 # the training target
    "label_defined": ts > 1,                   # False for the 90 rows with T_S <= 1
})

for col, fname in [("window_upstream",  "chen2013_window_up90.csv"),
                   ("window_up90_dn19", "chen2013_window_up90_dn19.csv")]:
    out = base.copy()
    out.insert(3, "window", df_full[col])
    out.to_csv(OUT / fname, index=False)
    print(f"{fname:<34} {out.shape[0]} rows x {out.shape[1]} cols, "
          f"window {out.window.str.len().iloc[0]} nt")

print()
print(f"label_defined: {int(base.label_defined.sum())} usable, "
      f"{int((~base.label_defined).sum())} with T_S <= 1 (kept, logit_te is NaN)")
print(f"distinct hairpins: {base.hairpin.nunique()} -- cluster these for the split, "
      f"do NOT split at random")

chen2013_window_up90.csv           582 rows x 8 cols, window 90 nt
chen2013_window_up90_dn19.csv      582 rows x 8 cols, window 109 nt

label_defined: 492 usable, 90 with T_S <= 1 (kept, logit_te is NaN)
distinct hairpins: 387 -- cluster these for the split, do NOT split at random


c:\venvs\TE_model\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
#manifold learning


$$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}} = 1 - \frac{4}{40} = 0.90$$

In [ ]:
# do log of 300

data = np.log()
print(data)

-1.2039728043259361
